# Advertisement Brain-Response Neuromarketing API — Fixed Clean Inference Notebook

This notebook exposes a FastAPI endpoint for analyzing an uploaded advertisement image using TRIBE v2 predicted brain-response features.

It is intentionally lean:

- no emotion classifier
- no CSV training helpers
- no joblib/pickle decoder loading
- no static `RESEARCH_MAPPING` JSON
- no PNG-only fallback endpoint
- no duplicate proxy-score paths

The output is limited to brain activation features, research-informed neuromarketing engagement indices, visualizations, and explicit limitations.


## 1. Setup paths

In [5]:
from pathlib import Path
import json
import os

KAGGLE_WORKING = Path('/kaggle/working')
if not KAGGLE_WORKING.exists():
    KAGGLE_WORKING = Path('/mnt/data/kaggle_working')
KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = KAGGLE_WORKING / 'results'
TEMP_DIR = KAGGLE_WORKING / 'ad_api_temp'
CACHE_DIR = KAGGLE_WORKING / 'model_cache'
for d in [RESULTS_DIR, TEMP_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Working directory:', KAGGLE_WORKING)
print('Results directory:', RESULTS_DIR)
print('Mode: clean no-training neuromarketing inference')


Working directory: /kaggle/working
Results directory: /kaggle/working/results
Mode: clean no-training neuromarketing inference


## 2. Install dependencies

This installs the API runtime, the official TRIBE v2 repository, and the known-compatible `exca==0.5.23` pin for the model-loading issue.


In [6]:
import os
import sys
import subprocess

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:64")
os.environ.setdefault("CUDA_MODULE_LOADING", "LAZY")
os.environ.setdefault("MALLOC_ARENA_MAX", "2")

def run_cmd(cmd, required=True, timeout=None):
    print("\n$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=timeout)
    if result.stdout:
        print(result.stdout[-6000:])
    if required and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(map(str, cmd))}")
    return result.returncode == 0

# ffmpeg is required for image -> silent MP4 conversion.
if subprocess.run(["bash", "-lc", "command -v ffmpeg"], stdout=subprocess.PIPE).returncode != 0:
    run_cmd(["bash", "-lc", "apt-get update -y && apt-get install -y ffmpeg"], required=False, timeout=300)
else:
    print("ffmpeg is available.")

# Base API/runtime utilities. Avoid broad `pip install -U`.
BASE_PACKAGES = [
    "uv",
    "fastapi>=0.115,<0.123",
    "uvicorn[standard]>=0.30,<0.39",
    "python-multipart>=0.0.9,<0.1",
    "pyngrok>=7,<8",
    "nest_asyncio>=1.6,<2",
    "requests>=2.31,<3",
    "pillow>=9,<12",
    "matplotlib==3.10.0",
    "numpy==2.2.6",
    "huggingface_hub>=0.24,<2",
    "imageio",
    "imageio-ffmpeg",
    "psutil>=5.9,<8",
    "nilearn>=0.12,<0.13",
]

run_cmd([
    sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "--upgrade-strategy", "only-if-needed",
    *BASE_PACKAGES,
], required=True, timeout=900)

# TRIBE v2 runtime dependencies from pyproject.toml, with the observed-compatible exca pin.
TRIBE_RUNTIME_DEPS = [
    "exca==0.5.23",
    "neuralset==0.0.2",
    "neuraltrain==0.0.2",
    "torch>=2.5.1,<2.7",
    "torchvision>=0.20,<0.22",
    "x_transformers==1.27.20",
    "einops",
    "pyyaml",
    "moviepy>=2.2.1",
    "gtts",
    "langdetect",
    "spacy",
    "soundfile",
    "Levenshtein",
    "julius",
    "transformers",
]

run_cmd([sys.executable, "-m", "uv", "pip", "install", "--system", *TRIBE_RUNTIME_DEPS], required=True, timeout=1200)

# Install the TRIBE repository itself without allowing dependency resolution to undo the compatible pins.
run_cmd([
    sys.executable, "-m", "uv", "pip", "install", "--system", "--no-deps",
    "git+https://github.com/facebookresearch/tribev2.git",
], required=True, timeout=1200)

# Re-apply the known-compatible exca version after all installs.
run_cmd([sys.executable, "-m", "uv", "pip", "install", "--system", "--force-reinstall", "exca==0.5.23"], required=True, timeout=600)

print("\nInstalled versions:")
for module_name in ["numpy", "matplotlib", "PIL", "fastapi", "uvicorn", "requests", "pyngrok", "torch"]:
    try:
        mod = __import__(module_name)
        print(f"{module_name}: {getattr(mod, '__version__', 'unknown')}")
    except Exception as exc:
        print(f"{module_name}: import failed: {exc!r}")

print("\nDependency installation complete.")
print("If this kernel was already polluted by previous broad upgrades, restart the Kaggle session and run all cells from the top.")


ffmpeg is available.

$ /usr/bin/python3 -m pip install -q --no-cache-dir --upgrade-strategy only-if-needed uv fastapi>=0.115,<0.123 uvicorn[standard]>=0.30,<0.39 python-multipart>=0.0.9,<0.1 pyngrok>=7,<8 nest_asyncio>=1.6,<2 requests>=2.31,<3 pillow>=9,<12 matplotlib==3.10.0 numpy==2.2.6 huggingface_hub>=0.24,<2 imageio imageio-ffmpeg psutil>=5.9,<8 nilearn>=0.12,<0.13
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 229.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is

## 3. Check runtime device

In [7]:
import json
import platform

DEVICE = "cpu"
GPU_INFO = {"cuda_available": False, "device": "cpu"}

try:
    import torch
    GPU_INFO["cuda_available"] = bool(torch.cuda.is_available())
    if torch.cuda.is_available():
        DEVICE = "cuda"
        GPU_INFO.update({
            "device": "cuda",
            "gpu_name": torch.cuda.get_device_name(0),
            "gpu_count": torch.cuda.device_count(),
        })
except Exception as exc:
    GPU_INFO["torch_error"] = repr(exc)

print(json.dumps({
    "python": platform.python_version(),
    "device": DEVICE,
    "gpu_info": GPU_INFO,
}, indent=2))

{
  "python": "3.12.13",
  "device": "cuda",
  "gpu_info": {
    "cuda_available": true,
    "device": "cuda",
    "gpu_name": "Tesla T4",
    "gpu_count": 2
  }
}


## 4. Load TRIBE v2 model

Uses the official loader only: `TribeModel.from_pretrained("facebook/tribev2", cache_folder=...)`. No fake fallback and no custom adapter path are used.


In [8]:
import os
import sys
import json
import gc
import inspect
import traceback
import importlib
from pathlib import Path
from typing import Any, Dict, Optional

# Important when rerunning the notebook after an OOM or after editing cells.
try:
    if "MODEL" in globals() and MODEL is not None:
        del MODEL
except Exception:
    pass
gc.collect()
try:
    import torch
    torch.set_grad_enabled(False)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception:
    pass

MODEL = None

MODEL_STATUS = {
    "loaded": False,
    "source": None,
    "error": None,
    "traceback": None,
    "device": DEVICE,
    "checkpoint": "facebook/tribev2",
    "cache_folder": str(CACHE_DIR),
    "from_pretrained_signature": None,
    "versions": {},
}

def get_secret_or_env(names):
    for name in names:
        value = os.environ.get(name)
        if value:
            return value, f"env:{name}"
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in names:
            try:
                value = client.get_secret(name)
                if value:
                    os.environ[name] = value
                    return value, f"kaggle_secret:{name}"
            except Exception:
                pass
    except Exception:
        pass
    return None, None

def package_version(pkg_name: str) -> str:
    try:
        from importlib.metadata import version
        return version(pkg_name)
    except Exception as exc:
        return f"unavailable: {exc!r}"

def collect_runtime_versions() -> dict:
    packages = [
        "tribev2",
        "exca",
        "neuralset",
        "neuraltrain",
        "torch",
        "torchvision",
        "numpy",
        "transformers",
        "huggingface_hub",
    ]
    return {name: package_version(name) for name in packages}

def verify_exca_novalue_available() -> None:
    """
    Fail fast if the old Kaggle error would occur again.
    This is not a fallback shim; it only verifies that dependency installation is correct.
    """
    base = importlib.import_module("exca.steps.base")
    if not hasattr(base, "NoValue"):
        raise RuntimeError(
            "exca.steps.base.NoValue is missing. "
            "Rerun the dependency cell; it pins exca==0.5.23 to avoid this model-loading error."
        )

# Hugging Face auth is optional and never hardcoded.
hf_token, hf_source = get_secret_or_env(["HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"])
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    try:
        from huggingface_hub import login, whoami
        login(token=hf_token, add_to_git_credential=False)
        who = whoami(token=hf_token)
        print("Hugging Face authenticated via", hf_source, "as", who.get("name") or who.get("fullname") or "authenticated_user")
    except Exception as exc:
        print("HF token was found, but login/whoami failed:", repr(exc))
else:
    print("No HF_TOKEN found. Loading may still work if files are public/cached, but HF_TOKEN is recommended.")

def _clear_model_load_memory():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass

def load_tribe_v2_model():
    global MODEL, MODEL_STATUS

    if MODEL is not None:
        MODEL_STATUS["loaded"] = True
        return MODEL

    MODEL_STATUS["versions"] = collect_runtime_versions()

    try:
        _clear_model_load_memory()
        verify_exca_novalue_available()

        from tribev2 import TribeModel
        MODEL_STATUS["source"] = "tribev2.TribeModel"

        try:
            MODEL_STATUS["from_pretrained_signature"] = str(inspect.signature(TribeModel.from_pretrained))
        except Exception:
            MODEL_STATUS["from_pretrained_signature"] = "unavailable"

        # Official README pattern. Do not pass device='auto'.
        MODEL = TribeModel.from_pretrained(
            "facebook/tribev2",
            cache_folder=str(CACHE_DIR),
        )

        # Inference API only. This avoids storing gradients in normal PyTorch modules.
        try:
            if hasattr(MODEL, "eval"):
                MODEL.eval()
        except Exception:
            pass

        _clear_model_load_memory()

        MODEL_STATUS.update({
            "loaded": True,
            "error": None,
            "traceback": None,
            "class": f"{MODEL.__class__.__module__}.{MODEL.__class__.__name__}",
        })

        print("TRIBE v2 loaded successfully.")
        print(json.dumps(MODEL_STATUS, indent=2))
        return MODEL

    except Exception as exc:
        MODEL = None
        MODEL_STATUS.update({
            "loaded": False,
            "error": repr(exc),
            "traceback": traceback.format_exc(),
        })
        print("TRIBE v2 failed to load.")
        print(json.dumps(MODEL_STATUS, indent=2))
        print("\nFull traceback:\n", MODEL_STATUS["traceback"])
        _clear_model_load_memory()
        return None

MODEL = load_tribe_v2_model()


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face authenticated via env:HF_TOKEN as Hasin1415


/usr/local/lib/python3.12/dist-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-06-09 12:54:53 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
INFO - Loading model from /root/.cache/huggingface/hub/models--facebook--tribev2/snapshots/f894e783020944dcd96e5568550afe2aa9743f9f/best.ckpt


TRIBE v2 loaded successfully.
{
  "loaded": true,
  "source": "tribev2.TribeModel",
  "error": null,
  "traceback": null,
  "device": "cuda",
  "checkpoint": "facebook/tribev2",
  "cache_folder": "/kaggle/working/model_cache",
  "from_pretrained_signature": "(checkpoint_dir: str | pathlib.Path, checkpoint_name: str = 'best.ckpt', cache_folder: str | pathlib.Path = None, cluster: str = None, device: str = 'auto', config_update: dict | None = None) -> 'TribeModel'",
  "versions": {
    "tribev2": "0.1.0",
    "exca": "0.5.23",
    "neuralset": "0.0.2",
    "neuraltrain": "0.0.2",
    "torch": "2.6.0",
    "torchvision": "0.21.0",
    "numpy": "2.4.6",
    "transformers": "5.0.0",
    "huggingface_hub": "1.10.1"
  },
  "class": "tribev2.demo_utils.TribeModel"
}


## 5. Utility functions

In [9]:
import re
import uuid
import shutil
import subprocess
import gc
import os
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
from PIL import Image, UnidentifiedImageError, ImageOps

SYSTEM_LABEL = "Predicted neural-response proxies derived from TRIBE v2 outputs."
ALLOWED_IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

# OOM-safety knobs. Override from Kaggle env/secrets if needed.
MAX_UPLOAD_MB = int(os.environ.get("MAX_UPLOAD_MB", "8"))
MAX_UPLOAD_BYTES = MAX_UPLOAD_MB * 1024 * 1024
# Hard safety cap: images under this cap are accepted and resized automatically if needed.
# Raise carefully only if your Kaggle RAM can decode very large images.
MAX_IMAGE_PIXELS = int(os.environ.get("MAX_IMAGE_PIXELS", "50000000"))   # 50 MP hard decode limit
MAX_IMAGE_SIDE = int(os.environ.get("MAX_IMAGE_SIDE", "512"))            # max long side used for inference
VIDEO_DURATION_SECONDS = float(os.environ.get("VIDEO_DURATION_SECONDS", "1.0"))
VIDEO_FPS = int(os.environ.get("VIDEO_FPS", "1"))

Image.MAX_IMAGE_PIXELS = MAX_IMAGE_PIXELS

def safe_stem(name: str, max_len: int = 80) -> str:
    stem = Path(name).stem or "upload"
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", stem).strip("._-") or "upload"
    return stem[:max_len]

def free_runtime_memory():
    """Best-effort cleanup between POST requests."""
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass

def memory_snapshot() -> Dict[str, Any]:
    info = {}
    try:
        import psutil
        vm = psutil.virtual_memory()
        info["cpu_ram"] = {
            "available_mb": round(vm.available / (1024**2), 1),
            "used_mb": round(vm.used / (1024**2), 1),
            "percent": vm.percent,
        }
    except Exception as exc:
        info["cpu_ram_error"] = repr(exc)
    try:
        import torch
        info["cuda_available"] = bool(torch.cuda.is_available())
        if torch.cuda.is_available():
            info["cuda"] = {
                "allocated_mb": round(torch.cuda.memory_allocated() / (1024**2), 1),
                "reserved_mb": round(torch.cuda.memory_reserved() / (1024**2), 1),
                "max_allocated_mb": round(torch.cuda.max_memory_allocated() / (1024**2), 1),
                "device": torch.cuda.get_device_name(0),
            }
    except Exception as exc:
        info["cuda_error"] = repr(exc)
    return info

def resized_dimensions(width: int, height: int, max_side: int = MAX_IMAGE_SIDE) -> Tuple[int, int, bool]:
    """Return inference dimensions. Resize only when the image is larger than max_side."""
    width, height, max_side = int(width), int(height), int(max_side)
    longest = max(width, height)
    if longest <= max_side:
        return width, height, False
    scale = max_side / float(longest)
    new_w = max(1, int(round(width * scale)))
    new_h = max(1, int(round(height * scale)))
    return new_w, new_h, True

def validate_image_file(path: Path) -> Dict[str, Any]:
    if not path.exists() or not path.is_file():
        raise ValueError(f"Image file does not exist: {path}")
    if path.suffix.lower() not in ALLOWED_IMAGE_SUFFIXES:
        raise ValueError(f"Unsupported image type '{path.suffix}'. Supported: {sorted(ALLOWED_IMAGE_SUFFIXES)}")
    if path.stat().st_size > MAX_UPLOAD_BYTES:
        raise ValueError(f"Uploaded file is too large: {path.stat().st_size / (1024**2):.2f} MB. Limit is {MAX_UPLOAD_MB} MB.")
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            width, height = img.size
            mode = img.mode
            fmt = img.format
    except UnidentifiedImageError as exc:
        raise ValueError("Uploaded file is not a valid image.") from exc
    except Exception as exc:
        raise ValueError(f"Image validation failed: {exc}") from exc
    if width < 16 or height < 16:
        raise ValueError(f"Image is too small: {width}x{height}")

    pixels = int(width) * int(height)
    if pixels > MAX_IMAGE_PIXELS:
        raise ValueError(
            f"Image has too many pixels to safely decode in this Kaggle runtime: {width}x{height} = {pixels:,}. "
            f"Hard limit is {MAX_IMAGE_PIXELS:,}. Upload a smaller image or raise MAX_IMAGE_PIXELS carefully."
        )

    inference_width, inference_height, resized_for_inference = resized_dimensions(width, height, MAX_IMAGE_SIDE)
    return {
        "original_width": width,
        "original_height": height,
        "original_pixels": pixels,
        "mode": mode,
        "format": fmt,
        "upload_size_mb": round(path.stat().st_size / (1024**2), 3),
        "resized_for_inference": resized_for_inference,
        "inference_width": inference_width,
        "inference_height": inference_height,
        "max_image_side_used_for_inference": MAX_IMAGE_SIDE,
        "hard_max_image_pixels": MAX_IMAGE_PIXELS,
    }

def normalize_image_to_rgb_png(src: Path, dst: Path, max_side: int = MAX_IMAGE_SIDE) -> Path:
    """
    Normalize and resize before video conversion.
    Smaller images remain at their original size; only images larger than max_side are downscaled.
    This prevents TRIBE/video preprocessing from seeing a huge frame during a POST request.
    """
    dst.parent.mkdir(parents=True, exist_ok=True)
    with Image.open(src) as img:
        img = ImageOps.exif_transpose(img)
        try:
            # Helps JPEG decoders reduce memory before full decode when possible.
            img.draft("RGB", (max_side, max_side))
        except Exception:
            pass
        img = img.convert("RGB")
        original_size = img.size
        if max(original_size) > max_side:
            img.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
        # Use optimized PNG for broad ffmpeg compatibility. The dimensions are now bounded.
        img.save(dst, format="PNG", optimize=True)
    return dst


def run_subprocess(cmd: List[str], timeout: int = 120) -> subprocess.CompletedProcess:
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=timeout)
    if result.returncode != 0:
        raise RuntimeError(
            "Command failed: " + " ".join(cmd) +
            "\nSTDOUT:\n" + result.stdout[-2000:] +
            "\nSTDERR:\n" + result.stderr[-4000:]
        )
    return result

def delete_path(path: Optional[Path]) -> bool:
    if path is None:
        return False
    try:
        path = Path(path)
        if not path.exists():
            return False
        if path.is_dir():
            shutil.rmtree(path, ignore_errors=True)
        else:
            path.unlink(missing_ok=True)
        return True
    except Exception:
        return False

def to_numpy(x: Any) -> np.ndarray:
    if x is None:
        return np.asarray([])
    if hasattr(x, "detach"):
        return x.detach().cpu().numpy()
    if hasattr(x, "cpu") and hasattr(x, "numpy"):
        return x.cpu().numpy()
    return np.asarray(x)

def robust_minmax(values: Dict[str, float]) -> Dict[str, float]:
    clean = {k: float(v) for k, v in values.items() if v is not None and np.isfinite(v)}
    if not clean:
        return {}
    arr = np.asarray(list(clean.values()), dtype=np.float32)
    lo, hi = np.nanpercentile(arr, [5, 95])
    if not np.isfinite(lo) or not np.isfinite(hi) or abs(hi - lo) < 1e-12:
        med = np.nanmedian(arr)
        mad = np.nanmedian(np.abs(arr - med)) + 1e-9
        return {k: float(np.clip(0.5 + 0.25 * ((v - med) / mad), 0.0, 1.0)) for k, v in clean.items()}
    return {k: float(np.clip((v - lo) / (hi - lo), 0.0, 1.0)) for k, v in clean.items()}

def weighted_mean(named_scores: Dict[str, float], weights: Dict[str, float], default: float = 0.0) -> Tuple[float, List[str]]:
    total, denom = 0.0, 0.0
    used = []
    for key, weight in weights.items():
        value = named_scores.get(key)
        if value is not None and np.isfinite(value):
            total += float(value) * float(weight)
            denom += float(weight)
            used.append(key)
    if denom == 0:
        return float(default), used
    return float(np.clip(total / denom, 0.0, 1.0)), used

def confidence_from_available(used: List[str], unavailable_subcortical: bool = False, base: str = "medium") -> str:
    if not used:
        return "low"
    if unavailable_subcortical:
        return "low" if len(used) < 2 else "medium"
    return base

print("Utilities ready with auto-resize + OOM-safe limits:")
print({
    "MAX_UPLOAD_MB": MAX_UPLOAD_MB,
    "MAX_IMAGE_PIXELS": MAX_IMAGE_PIXELS,
    "MAX_IMAGE_SIDE": MAX_IMAGE_SIDE,
    "VIDEO_DURATION_SECONDS": VIDEO_DURATION_SECONDS,
    "VIDEO_FPS": VIDEO_FPS,
})


Utilities ready with auto-resize + OOM-safe limits:
{'MAX_UPLOAD_MB': 8, 'MAX_IMAGE_PIXELS': 50000000, 'MAX_IMAGE_SIDE': 512, 'VIDEO_DURATION_SECONDS': 1.0, 'VIDEO_FPS': 1}


## 6. Image-to-video conversion and TRIBE video inference

Uploaded static images are validated, converted to a one-second silent MP4, and passed to TRIBE through `video_path` only. Direct image-model inference branches are intentionally removed.


In [10]:
def image_to_silent_video(src_path: Path, output_dir: Path, duration: float = VIDEO_DURATION_SECONDS, fps: int = VIDEO_FPS) -> Path:
    """Convert an uploaded image into a tiny silent MP4 after auto-resizing large images."""
    output_dir.mkdir(parents=True, exist_ok=True)
    normalized_png = output_dir / f"{safe_stem(src_path.name)}_{uuid.uuid4().hex[:8]}_auto_resized_rgb.png"
    normalize_image_to_rgb_png(src_path, normalized_png, max_side=MAX_IMAGE_SIDE)

    video_path = output_dir / f"{safe_stem(src_path.name)}_{uuid.uuid4().hex[:8]}_silent.mp4"
    cmd = [
        "ffmpeg", "-y",
        "-hide_banner", "-loglevel", "error",
        "-threads", "1",
        "-loop", "1",
        "-i", str(normalized_png),
        "-f", "lavfi",
        "-i", "anullsrc=channel_layout=mono:sample_rate=16000",
        "-t", str(duration),
        "-r", str(fps),
        "-shortest",
        "-vf", f"scale='min({MAX_IMAGE_SIDE},iw)':'min({MAX_IMAGE_SIDE},ih)':force_original_aspect_ratio=decrease,pad=ceil(iw/2)*2:ceil(ih/2)*2",
        "-c:v", "libx264",
        "-preset", "ultrafast",
        "-pix_fmt", "yuv420p",
        "-c:a", "aac",
        "-b:a", "32k",
        "-movflags", "+faststart",
        str(video_path),
    ]
    try:
        run_subprocess(cmd, timeout=120)
    finally:
        delete_path(normalized_png)
        free_runtime_memory()

    if not video_path.exists() or video_path.stat().st_size == 0:
        raise RuntimeError("Image-to-video conversion produced no output.")
    return video_path

def _extract_prediction_payload(raw_output: Any) -> Any:
    if isinstance(raw_output, tuple) and raw_output:
        raw_output = raw_output[0]
    if isinstance(raw_output, dict):
        for key in ["activations", "predictions", "preds", "stat_map", "fmri", "brain_response"]:
            if key in raw_output:
                return raw_output[key]
        return None
    return raw_output

def prediction_shape_and_size(raw_output: Any) -> Tuple[List[int], int]:
    """
    Read prediction metadata without copying large tensors/arrays to NumPy.
    The previous notebook converted the whole output just to get shape, which can OOM.
    """
    x = _extract_prediction_payload(raw_output)
    if x is None:
        return [], 0
    shape = getattr(x, "shape", None)
    if shape is None:
        try:
            arr = np.asarray(x)
            return list(arr.shape), int(arr.size)
        except Exception:
            return [], 0
    shape = [int(s) for s in shape]
    size = 1
    for s in shape:
        size *= int(s)
    return shape, int(size)

def aggregate_activation(preds: Any) -> np.ndarray:
    """
    Aggregate predictions to one vector while avoiding full tensor -> NumPy copies.
    If `preds` is a torch tensor, reduction happens before moving to CPU.
    """
    try:
        import torch
        if torch.is_tensor(preds):
            with torch.inference_mode():
                t = preds.detach()
                if not torch.is_floating_point(t):
                    t = t.float()
                elif t.dtype not in (torch.float16, torch.bfloat16, torch.float32):
                    t = t.float()

                if t.numel() == 0:
                    return np.asarray([], dtype=np.float32)
                if t.ndim == 1:
                    stat = t
                elif t.ndim == 2:
                    stat = torch.nanmean(t, dim=0)
                elif t.ndim == 3:
                    stat = torch.nanmean(t.reshape(-1, t.shape[-1]), dim=0)
                else:
                    stat = t.reshape(-1)

                stat = torch.nan_to_num(stat, nan=0.0, posinf=0.0, neginf=0.0)
                out = stat.float().cpu().numpy().astype(np.float32, copy=False)
                del t, stat
                free_runtime_memory()
                return out
    except Exception:
        pass

    arr = to_numpy(preds)
    if arr.size == 0:
        return np.asarray([], dtype=np.float32)
    arr = arr.astype(np.float32, copy=False)
    if arr.ndim == 1:
        stat = arr
    elif arr.ndim == 2:
        stat = np.nanmean(arr, axis=0, dtype=np.float32)
    elif arr.ndim == 3:
        stat = np.nanmean(arr.reshape(-1, arr.shape[-1]), axis=0, dtype=np.float32)
    else:
        stat = arr.reshape(-1)
    stat = np.nan_to_num(stat, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)
    return stat

def _predict_with_inference_mode(events):
    try:
        import torch
        with torch.inference_mode():
            return MODEL.predict(events=events)
    except ImportError:
        return MODEL.predict(events=events)

def run_tribe_video_inference(video_path: Path) -> Dict[str, Any]:
    """Run TRIBE through the public video event path only."""
    if MODEL is None:
        raise RuntimeError(MODEL_STATUS.get("error") or "TRIBE v2 model is unavailable.")
    if not (hasattr(MODEL, "get_events_dataframe") and hasattr(MODEL, "predict")):
        raise RuntimeError("Loaded TRIBE model does not expose get_events_dataframe(...) and predict(...).")

    events = None
    try:
        events = MODEL.get_events_dataframe(video_path=str(video_path))
        events_count = int(len(events)) if hasattr(events, "__len__") else None

        output = _predict_with_inference_mode(events=events)
        prediction_shape, prediction_size = prediction_shape_and_size(output)
        if prediction_size == 0:
            raise RuntimeError("TRIBE returned an empty prediction array.")

        return {
            "raw_output": output,
            "processed_as": f"{VIDEO_DURATION_SECONDS:g}_second_silent_video_{VIDEO_FPS}_fps_{MAX_IMAGE_SIDE}px_max",
            "prediction_shape": prediction_shape,
            "events_count": events_count,
        }
    finally:
        # Do not keep the events dataframe in memory after predict.
        try:
            del events
        except Exception:
            pass
        free_runtime_memory()

def run_tribe_inference_for_upload(ad_file_path: Path, run_dir: Path) -> Dict[str, Any]:
    if MODEL is None or not MODEL_STATUS.get("loaded"):
        raise RuntimeError(f"TRIBE v2 model is unavailable: {MODEL_STATUS.get('error')}")

    processing_dir = run_dir / "processing"
    processing_dir.mkdir(parents=True, exist_ok=True)
    temporary_video_path = None

    try:
        temporary_video_path = image_to_silent_video(ad_file_path, processing_dir)
        result = run_tribe_video_inference(temporary_video_path)
        return {
            "raw_output": result.get("raw_output"),
            "processed_as": result.get("processed_as", "1_second_silent_video"),
            "temporary_video_path": temporary_video_path,
            "prediction_shape": result.get("prediction_shape"),
            "events_count": result.get("events_count"),
        }
    except Exception as exc:
        free_runtime_memory()
        if exc.__class__.__name__ == "OutOfMemoryError" or "out of memory" in str(exc).lower() or "cuda" in str(exc).lower():
            raise RuntimeError(
                "TRIBE inference failed due to memory pressure. "
                f"Current limits: MAX_IMAGE_SIDE={MAX_IMAGE_SIDE}, VIDEO_FPS={VIDEO_FPS}, duration={VIDEO_DURATION_SECONDS}s. "
                "Restart Kaggle, rerun all cells, and if it still fails set MAX_IMAGE_SIDE=384 before running the notebook."
            ) from exc
        raise RuntimeError(f"TRIBE inference failed: {repr(exc)}") from exc
    finally:
        free_runtime_memory()

print("OOM-safe video-only inference pipeline ready. Large uploaded images are auto-resized before silent MP4 conversion.")

OOM-safe video-only inference pipeline ready. Large uploaded images are auto-resized before silent MP4 conversion.


## 7. ROI mapping and proxy scoring

In [11]:
REGION_ALIASES = {
    "visual_cortex": [
        "visual", "occip", "calcar", "calcarine", "cune", "cuneus", "lingual",
        "v1", "v2", "v3", "v4", "pericalcarine",
    ],
    "ventral_visual_object_form": [
        "fusiform", "object", "form", "inferior_temporal", "temporal_inf", "temporal_inferior",
        "ventral_visual", "occipito_temporal", "occipito_temporal", "occipitotemporal", "oc_temp",
    ],
    "dorsal_attention_parietal": [
        "pariet", "intrapar", "intraparietal", "superior_parietal", "inferior_parietal",
        "precuneus", "angular", "supramarginal", "postcentral",
    ],
    "prefrontal_control": [
        "prefrontal", "dlpfc", "frontal_sup", "frontal_middle", "middle_frontal",
        "superior_frontal", "inferior_frontal", "front_sup", "front_middle", "front_inf",
        "frontomarginal", "fronto_marginal",
    ],
    "orbitofrontal_vmPFC": [
        "orbitofrontal", "orbital", "ofc", "vmpfc", "vm_pfc", "medial_prefrontal",
        "medial_frontal", "front_orb", "g_rectus", "rectus", "subcallosal", "subcallos",
    ],
    "anterior_cingulate": [
        "anterior_cingulate", "cingulate_ant", "cingul_ant", "acc", "cingulum_ant",
    ],
    "posterior_cingulate_precuneus": [
        "posterior_cingulate", "cingulate_post", "cingul_post", "pcc", "cingulum_post", "precuneus",
    ],
    "insula_salience": ["insula", "insular"],
    "temporal_semantic_social": [
        "temporal", "sts", "superior_temporal", "temporal_sup", "temporal_pole", "pole_temporal",
        "semantic", "social", "planum", "transverse_temporal",
    ],
    "parahippocampal_memory_proxy": [
        "parahippocampal", "para_hippocampal", "entorhinal", "perirhinal", "collateral",
        "scene", "lingual", "fusiform",
    ],
    "motor_arousal_proxy": ["precentral", "postcentral", "motor", "supp_motor", "supplementary_motor", "paracentral"],
    "amygdala": ["amygdala"],
    "hippocampus": ["hippocampus"],
    "ventral_striatum_nucleus_accumbens": ["ventral_striatum", "nucleus_accumbens", "accumbens", "nacc"],
}

REGION_INTERPRETATIONS = {
    "visual_cortex": "visual salience / early visual engagement",
    "ventral_visual_object_form": "object/form and brand-visual processing proxy",
    "dorsal_attention_parietal": "visuospatial attention proxy",
    "prefrontal_control": "cognitive-control and executive-attention proxy",
    "orbitofrontal_vmPFC": "cortical valuation/reward proxy",
    "anterior_cingulate": "salience, conflict, arousal/action-preparation proxy",
    "posterior_cingulate_precuneus": "context/self-relevance/memory-network proxy",
    "insula_salience": "interoceptive/affective salience proxy",
    "temporal_semantic_social": "semantic/social/association proxy",
    "parahippocampal_memory_proxy": "scene/context memory encoding proxy",
    "motor_arousal_proxy": "action-preparation/arousal proxy",
    "amygdala": "subcortical affective-salience region",
    "hippocampus": "subcortical/medial-temporal memory region",
    "ventral_striatum_nucleus_accumbens": "subcortical reward/motivation region",
}

SURFACE_PROFILE_INTERPRETATIONS = {
    "surface_global_magnitude": "unlabeled whole-cortex predicted-response magnitude",
    "surface_high_activation_tail": "unlabeled high-activation tail strength across cortical vertices",
    "surface_activation_contrast": "unlabeled contrast between high-response and typical cortical vertices",
    "surface_signal_complexity": "unlabeled cortical response variability/complexity proxy",
    "surface_positive_balance": "unlabeled signed-response balance proxy",
    "surface_activation_sparsity": "unlabeled concentration/sparsity of stronger cortical responses",
}

SUBCORTICAL_REGIONS = {"amygdala", "hippocampus", "ventral_striatum_nucleus_accumbens"}

_DESTRIEUX_ATLAS_CACHE = None

def canonicalize_region_name(name: str) -> Optional[str]:
    low = re.sub(r"[^a-z0-9]+", "_", str(name).lower()).strip("_")
    for canonical, aliases in REGION_ALIASES.items():
        if canonical.lower() == low:
            return canonical
        for alias in aliases:
            alias_low = re.sub(r"[^a-z0-9]+", "_", str(alias).lower()).strip("_")
            if alias_low and alias_low in low:
                return canonical
    return None

def _to_clean_1d_float(x: Any) -> np.ndarray:
    arr = np.asarray(x, dtype=np.float32).reshape(-1)
    if arr.size == 0:
        return arr
    arr = arr[np.isfinite(arr)]
    return arr.astype(np.float32, copy=False)

def extract_region_scores_from_output(raw_output: Any) -> Tuple[Dict[str, float], Dict[str, Any], Optional[np.ndarray]]:
    metadata = {"source": "unknown"}
    stat_map = None
    region_scores = {}

    # Public model predict often returns (preds, segments). Keep diagnostics; use preds for activation.
    if isinstance(raw_output, tuple) and raw_output:
        metadata["source"] = "tuple_output"
        metadata["tuple_length"] = len(raw_output)
        if len(raw_output) > 1:
            metadata["second_item_type"] = type(raw_output[1]).__name__
            try:
                metadata["second_item_len"] = int(len(raw_output[1]))
            except Exception:
                pass
        raw_output = raw_output[0]

    if isinstance(raw_output, dict):
        metadata["source"] = metadata.get("source") if metadata.get("source") != "unknown" else "dict_output"
        metadata["dict_keys"] = list(raw_output.keys())[:30]
        # Accept explicit region scores from a custom adapter.
        for key in ["brain_region_scores", "region_scores", "roi_scores"]:
            if key in raw_output:
                items = raw_output[key]
                metadata["explicit_region_score_key"] = key
                if isinstance(items, dict):
                    for region, value in items.items():
                        c = canonicalize_region_name(region) or str(region)
                        try:
                            region_scores[c] = float(value)
                        except Exception:
                            pass
                elif isinstance(items, list):
                    for item in items:
                        if not isinstance(item, dict):
                            continue
                        region = item.get("region") or item.get("brain_region") or item.get("name")
                        value = item.get("activation", item.get("score", item.get("value")))
                        if region is None or value is None:
                            continue
                        c = canonicalize_region_name(region) or str(region)
                        try:
                            region_scores[c] = float(value)
                        except Exception:
                            pass
        # Accept arrays under common keys.
        for key in ["activations", "predictions", "preds", "stat_map", "fmri", "brain_response"]:
            if key in raw_output:
                metadata["activation_key"] = key
                stat_map = aggregate_activation(raw_output[key])
                break
    else:
        if metadata.get("source") == "unknown":
            metadata["source"] = "array_or_tensor_output"
        stat_map = aggregate_activation(raw_output)

    if stat_map is not None:
        try:
            metadata["stat_map_size"] = int(np.asarray(stat_map).size)
        except Exception:
            pass
    return region_scores, metadata, stat_map

def _decode_label(label: Any) -> str:
    if isinstance(label, bytes):
        return label.decode("utf-8", errors="ignore")
    return str(label)

def get_destrieux_surface_atlas() -> Tuple[Optional[Any], Dict[str, Any]]:
    """
    Optional real cortical atlas path. This avoids the old single-global fallback when
    TRIBE returns fsaverage5-like vertex predictions but no ROI table.
    """
    global _DESTRIEUX_ATLAS_CACHE
    meta = {"atlas": "nilearn.datasets.fetch_atlas_surf_destrieux", "available": False}

    if os.environ.get("ENABLE_NILEARN_ATLAS", "1").strip().lower() in {"0", "false", "no"}:
        meta["error"] = "disabled by ENABLE_NILEARN_ATLAS"
        return None, meta

    if _DESTRIEUX_ATLAS_CACHE is not None:
        meta["available"] = True
        meta["cached"] = True
        return _DESTRIEUX_ATLAS_CACHE, meta

    try:
        from nilearn import datasets
        data_dir = str(CACHE_DIR / "nilearn_data") if "CACHE_DIR" in globals() else None
        atlas = datasets.fetch_atlas_surf_destrieux(data_dir=data_dir, verbose=0)
        _DESTRIEUX_ATLAS_CACHE = atlas
        meta["available"] = True
        meta["cached"] = False
        return atlas, meta
    except Exception as exc:
        meta["error"] = repr(exc)
        return None, meta

def atlas_roi_scores_from_stat_map(stat_map: np.ndarray) -> Tuple[Dict[str, float], Dict[str, Any]]:
    """
    Convert unlabeled fsaverage5-like cortical vertices into broad ROI scores using
    Nilearn's Destrieux surface atlas when possible. If unavailable or shape-mismatched,
    return no scores and let the safer surface-profile fallback handle it.
    """
    stat = _to_clean_1d_float(stat_map)
    meta = {"method": "destrieux_surface_atlas", "available": False, "stat_map_size": int(stat.size)}
    if stat.size == 0:
        meta["error"] = "empty_stat_map"
        return {}, meta

    atlas, atlas_meta = get_destrieux_surface_atlas()
    meta["atlas_meta"] = atlas_meta
    if atlas is None:
        meta["error"] = atlas_meta.get("error", "atlas_unavailable")
        return {}, meta

    try:
        labels = [_decode_label(x) for x in list(atlas["labels"])]
        left_map = np.asarray(atlas["map_left"]).reshape(-1)
        right_map = np.asarray(atlas["map_right"]).reshape(-1)
    except Exception as exc:
        meta["error"] = f"atlas_read_failed: {exc!r}"
        return {}, meta

    n_left, n_right = int(left_map.size), int(right_map.size)
    meta["atlas_left_vertices"] = n_left
    meta["atlas_right_vertices"] = n_right

    hemispheres = []
    if stat.size == n_left + n_right:
        hemispheres = [("left", stat[:n_left], left_map), ("right", stat[n_left:n_left + n_right], right_map)]
        meta["shape_match"] = "left_plus_right"
    elif stat.size == n_left:
        hemispheres = [("left_or_single", stat, left_map)]
        meta["shape_match"] = "single_hemi_left_size"
    elif stat.size == n_right:
        hemispheres = [("right_or_single", stat, right_map)]
        meta["shape_match"] = "single_hemi_right_size"
    else:
        meta["error"] = f"stat_map_size_{stat.size}_does_not_match_destrieux_{n_left}+{n_right}"
        return {}, meta

    collectors: Dict[str, List[float]] = {}
    matched_labels: Dict[str, List[str]] = {}
    for hemi_name, hemi_stat, hemi_labels in hemispheres:
        if hemi_stat.size != hemi_labels.size:
            continue
        for label_id in np.unique(hemi_labels):
            try:
                label_int = int(label_id)
            except Exception:
                continue
            if label_int < 0:
                continue
            label_name = labels[label_int] if label_int < len(labels) else str(label_int)
            canonical = canonicalize_region_name(label_name)
            if canonical is None or canonical in SUBCORTICAL_REGIONS:
                continue
            mask = hemi_labels == label_id
            if not np.any(mask):
                continue
            vals = hemi_stat[mask]
            vals = vals[np.isfinite(vals)]
            if vals.size == 0:
                continue
            collectors.setdefault(canonical, []).append(float(np.nanmean(np.abs(vals))))
            matched_labels.setdefault(canonical, []).append(f"{hemi_name}:{label_name}")

    if not collectors:
        meta["error"] = "no_aliases_matched_atlas_labels"
        return {}, meta

    scores = {region: float(np.mean(vals)) for region, vals in collectors.items() if vals}
    meta["available"] = True
    meta["matched_regions"] = sorted(scores)
    meta["matched_labels_sample"] = {k: v[:8] for k, v in matched_labels.items()}
    return scores, meta

def surface_profile_scores_from_stat_map(stat_map: np.ndarray) -> Tuple[Dict[str, float], Dict[str, Any]]:
    """
    Last-resort non-ROI fallback. This is not a brain-region atlas. It prevents the
    misleading old behavior where every emotion reused the same global mean.
    """
    stat = _to_clean_1d_float(stat_map)
    meta = {"method": "unlabeled_surface_profile", "available": False, "stat_map_size": int(stat.size)}
    if stat.size == 0:
        meta["error"] = "empty_stat_map"
        return {}, meta

    abs_v = np.abs(stat).astype(np.float32, copy=False)
    eps = 1e-8
    mean_abs = float(np.mean(abs_v))
    std_abs = float(np.std(abs_v))
    signed_mean = float(np.mean(stat))
    p50, p75, p90, p95 = [float(x) for x in np.percentile(abs_v, [50, 75, 90, 95])]
    l1 = float(np.sum(abs_v)) + eps
    l2 = float(np.sqrt(np.sum(abs_v * abs_v))) + eps
    n = float(abs_v.size)
    hoyer_sparsity = (np.sqrt(n) - (l1 / l2)) / (np.sqrt(n) - 1.0 + eps)

    # Keep features in 0..1. These are different signal-shape summaries, not anatomy.
    global_magnitude = float(np.clip(np.tanh(mean_abs + std_abs), 0.0, 1.0))
    high_tail = float(np.clip(np.tanh(p95 + 0.5 * p90), 0.0, 1.0))
    contrast = float(np.clip((p95 - p50) / (p95 + p50 + eps), 0.0, 1.0))
    complexity = float(np.clip(std_abs / (mean_abs + std_abs + eps), 0.0, 1.0))
    positive_balance = float(np.clip(0.5 + 0.5 * (signed_mean / (mean_abs + eps)), 0.0, 1.0))
    sparsity = float(np.clip(hoyer_sparsity, 0.0, 1.0))

    scores = {
        "surface_global_magnitude": global_magnitude,
        "surface_high_activation_tail": high_tail,
        "surface_activation_contrast": contrast,
        "surface_signal_complexity": complexity,
        "surface_positive_balance": positive_balance,
        "surface_activation_sparsity": sparsity,
    }
    meta.update({
        "available": True,
        "raw_mean_abs": mean_abs,
        "raw_std_abs": std_abs,
        "raw_signed_mean": signed_mean,
        "raw_percentiles_abs": {"p50": p50, "p75": p75, "p90": p90, "p95": p95},
        "note": "No atlas/ROI labels were available; these are unlabeled cortical surface signal summaries.",
    })
    return scores, meta

def _region_details_from_scores(raw: Dict[str, float], normalized: Dict[str, float], scoring_mode: str) -> List[Dict[str, Any]]:
    details = []
    if scoring_mode == "unlabeled_surface_profile":
        for key in SURFACE_PROFILE_INTERPRETATIONS:
            if key in normalized:
                details.append({
                    "region": key,
                    "available": True,
                    "activation": round(float(np.clip(normalized[key], 0, 1)), 4),
                    "raw_activation": float(raw[key]),
                    "confidence": "low",
                    "interpretation": SURFACE_PROFILE_INTERPRETATIONS[key],
                })
        details.append({
            "region": "global_cortical_surface",
            "available": True,
            "activation": round(float(normalized.get("surface_global_magnitude", 0.0)), 4),
            "raw_activation": float(raw.get("surface_global_magnitude", 0.0)),
            "confidence": "low",
            "interpretation": "global cortical activity proxy derived from an unlabeled surface vector",
        })
        for region in SUBCORTICAL_REGIONS:
            details.append({
                "region": region,
                "available": False,
                "activation": None,
                "confidence": "low",
                "interpretation": REGION_INTERPRETATIONS.get(region, region) + " unavailable without explicit subcortical ROI output",
            })
        return details

    for region in REGION_ALIASES:
        if region in normalized:
            details.append({
                "region": region,
                "available": True,
                "activation": round(float(np.clip(normalized[region], 0, 1)), 4),
                "raw_activation": float(raw[region]),
                "confidence": "medium" if region not in SUBCORTICAL_REGIONS else "medium",
                "interpretation": REGION_INTERPRETATIONS.get(region, region),
            })
        else:
            details.append({
                "region": region,
                "available": False,
                "activation": None,
                "confidence": "low",
                "interpretation": REGION_INTERPRETATIONS.get(region, region),
            })
    return details

def compute_roi_scores(raw_output: Any, normalize_scores: bool = True) -> Dict[str, Any]:
    explicit_scores, metadata, stat_map = extract_region_scores_from_output(raw_output)

    raw: Dict[str, float] = {}
    scoring_mode = "unknown"
    raw_already_normalized = False

    for region in REGION_ALIASES:
        if region in explicit_scores:
            val = abs(float(explicit_scores[region]))
            if np.isfinite(val):
                raw[region] = val

    if raw:
        scoring_mode = "explicit_region_scores"
        metadata["roi_extraction"] = {"method": scoring_mode, "available": True, "matched_regions": sorted(raw)}
    elif stat_map is not None and np.asarray(stat_map).size:
        atlas_scores, atlas_meta = atlas_roi_scores_from_stat_map(stat_map)
        metadata["atlas_roi_extraction"] = atlas_meta
        if atlas_scores:
            raw.update({k: abs(float(v)) for k, v in atlas_scores.items() if np.isfinite(v)})
            scoring_mode = "destrieux_surface_atlas"
        else:
            profile_scores, profile_meta = surface_profile_scores_from_stat_map(stat_map)
            metadata["surface_profile_extraction"] = profile_meta
            if profile_scores:
                raw.update(profile_scores)
                scoring_mode = "unlabeled_surface_profile"
                raw_already_normalized = True
            else:
                raise RuntimeError("TRIBE output had a stat map, but no atlas ROI or surface-profile scores could be computed.")
    else:
        raise RuntimeError("TRIBE output did not contain usable activations or region scores.")

    if raw_already_normalized:
        normalized = {k: float(np.clip(v, 0.0, 1.0)) for k, v in raw.items()}
    else:
        normalized = robust_minmax(raw) if normalize_scores else raw

    details = _region_details_from_scores(raw, normalized, scoring_mode=scoring_mode)
    score_lookup = {x["region"]: x.get("activation") for x in details if x.get("activation") is not None}

    return {
        "metadata": metadata,
        "scoring_mode": scoring_mode,
        "region_scores": details,
        "score_lookup": score_lookup,
        "stat_map": stat_map,
    }

BRAIN_FEATURE_WARNING_TEXT = (
    "This no-training version extracts TRIBE-predicted brain-response features and "
    "computes research-informed neuromarketing proxy indices. It does not perform "
    "emotion detection, does not load a trained decoder, and does not use a hidden "
    "research-mapping JSON file."
)

BRAIN_FEATURE_KEYS = [
    # ROI / atlas-derived features
    "visual_cortex",
    "ventral_visual_object_form",
    "dorsal_attention_parietal",
    "prefrontal_control",
    "orbitofrontal_vmPFC",
    "anterior_cingulate",
    "posterior_cingulate_precuneus",
    "insula_salience",
    "temporal_semantic_social",
    "parahippocampal_memory_proxy",
    "motor_arousal_proxy",
    "amygdala",
    "hippocampus",
    "ventral_striatum_nucleus_accumbens",
    # unlabeled surface profile features
    "surface_global_magnitude",
    "surface_high_activation_tail",
    "surface_activation_contrast",
    "surface_signal_complexity",
    "surface_positive_balance",
    "surface_activation_sparsity",
    # whole-feature descriptive statistics
    "feature_mean",
    "feature_std",
    "feature_min",
    "feature_max",
    "feature_range",
    "feature_energy",
    "feature_entropy",
    "feature_nonzero_ratio",
    # scoring-mode indicators
    "mode_explicit_region_scores",
    "mode_destrieux_surface_atlas",
    "mode_unlabeled_surface_profile",
]

def extract_brain_feature_dict(roi_result: Dict[str, Any]) -> Dict[str, float]:
    """
    Convert ROI/surface scores into a fixed numeric feature dictionary.
    This is feature extraction only; it does not infer emotions or train anything.
    """
    score_lookup = dict(roi_result.get("score_lookup", {}) or {})
    features: Dict[str, float] = {}
    for key in BRAIN_FEATURE_KEYS:
        if key.startswith("feature_") or key.startswith("mode_"):
            continue
        value = score_lookup.get(key, 0.0)
        try:
            value = float(value)
            features[key] = float(np.clip(value, 0.0, 1.0)) if np.isfinite(value) else 0.0
        except Exception:
            features[key] = 0.0

    base_keys = [k for k in BRAIN_FEATURE_KEYS if not k.startswith("feature_") and not k.startswith("mode_")]
    base_values = np.asarray([features.get(k, 0.0) for k in base_keys], dtype=np.float32)
    base_values = np.nan_to_num(base_values, nan=0.0, posinf=1.0, neginf=0.0)
    nonzero = base_values[base_values > 1e-8]
    p = np.abs(base_values)
    p_sum = float(p.sum()) + 1e-9
    p = p / p_sum
    entropy = float(-(p * np.log(p + 1e-9)).sum() / np.log(len(p) + 1e-9)) if len(p) else 0.0

    features.update({
        "feature_mean": float(np.clip(base_values.mean() if base_values.size else 0.0, 0.0, 1.0)),
        "feature_std": float(np.clip(base_values.std() if base_values.size else 0.0, 0.0, 1.0)),
        "feature_min": float(np.clip(base_values.min() if base_values.size else 0.0, 0.0, 1.0)),
        "feature_max": float(np.clip(base_values.max() if base_values.size else 0.0, 0.0, 1.0)),
        "feature_range": float(np.clip((base_values.max() - base_values.min()) if base_values.size else 0.0, 0.0, 1.0)),
        "feature_energy": float(np.clip(np.sqrt(np.mean(base_values * base_values)) if base_values.size else 0.0, 0.0, 1.0)),
        "feature_entropy": float(np.clip(entropy, 0.0, 1.0)),
        "feature_nonzero_ratio": float(np.clip(len(nonzero) / max(1, len(base_values)), 0.0, 1.0)),
    })

    mode = str(roi_result.get("scoring_mode", "unknown"))
    features["mode_explicit_region_scores"] = 1.0 if mode == "explicit_region_scores" else 0.0
    features["mode_destrieux_surface_atlas"] = 1.0 if mode == "destrieux_surface_atlas" else 0.0
    features["mode_unlabeled_surface_profile"] = 1.0 if mode == "unlabeled_surface_profile" else 0.0

    return {k: float(features.get(k, 0.0)) for k in BRAIN_FEATURE_KEYS}

def summarize_brain_features(feature_dict: Dict[str, float], top_k: int = 8) -> Dict[str, Any]:
    excluded_prefixes = ("feature_", "mode_")
    active = [
        {"feature": k, "value": round(float(v), 4)}
        for k, v in feature_dict.items()
        if not k.startswith(excluded_prefixes) and float(v) > 0
    ]
    active = sorted(active, key=lambda x: float(x["value"]), reverse=True)[:max(1, int(top_k or 8))]
    return {
        "top_active_features": active,
        "global_intensity": round(float(feature_dict.get("feature_mean", 0.0)), 4),
        "activation_contrast": round(float(feature_dict.get("feature_range", 0.0)), 4),
        "activation_dispersion": round(float(feature_dict.get("feature_std", 0.0)), 4),
        "feature_entropy": round(float(feature_dict.get("feature_entropy", 0.0)), 4),
        "nonzero_feature_ratio": round(float(feature_dict.get("feature_nonzero_ratio", 0.0)), 4),
        "basis": "Numeric features extracted from TRIBE-predicted brain response. No emotion labels, decoder, or training path is used.",
    }

NEUROMARKETING_EVIDENCE = [
    {
        "paper": "Venkatraman et al. (2015), Journal of Marketing Research, DOI 10.1509/jmr.13.0593",
        "use_in_notebook": "Supports attention/affect/memory/desirability as advertising constructs and notes fMRI, especially ventral striatum, can add predictive signal for market response.",
    },
    {
        "paper": "Falk, Berkman & Lieberman (2012), Psychological Science, DOI 10.1177/0956797611434964",
        "use_in_notebook": "Supports medial prefrontal/self-relevance systems as population-response predictors in persuasive media.",
    },
    {
        "paper": "Chan et al. (2024), Journal of Marketing Research, DOI 10.1177/00222437231194319",
        "use_in_notebook": "Supports reporting psychological-process domains: information processing, attention/executive function/memory, and social-affective response.",
    },
    {
        "paper": "Guixeres et al. (2017), Frontiers in Psychology, DOI 10.3389/fpsyg.2017.01808",
        "use_in_notebook": "Supports combining neuroscience-based metrics with machine learning to predict recall, liking, and online viewing outcomes.",
    },
]

NEUROMARKETING_WARNING_TEXT = (
    "Neuromarketing indices are research-informed proxy formulas over predicted brain-response features. "
    "They are not a validated emotion reader, not observed fMRI, and not a guaranteed KPI forecast. "
    "For production claims, validate these proxy indices against real campaign outcomes, A/B tests, or human studies."
)

NEUROMARKETING_INDEX_SPECS = {
    "attention_capture": {
        "description": "How strongly the stimulus may capture visual and attentional processing.",
        "engagement_type": "attention / stopping power",
        "weights": {
            "visual_cortex": 0.30,
            "dorsal_attention_parietal": 0.25,
            "anterior_cingulate": 0.15,
            "surface_high_activation_tail": 0.15,
            "surface_activation_contrast": 0.15,
        },
        "critical_features": ["visual_cortex", "dorsal_attention_parietal"],
        "research_basis": "Ad-processing research commonly treats attention/perception as core ad-effectiveness processes; this index uses visual and frontoparietal/cingulate proxies rather than original image features.",
    },
    "desirability_reward": {
        "description": "Potential desire, valuation, wanting, or conversion pull.",
        "engagement_type": "desire / purchase intent proxy",
        "weights": {
            "ventral_striatum_nucleus_accumbens": 0.35,
            "orbitofrontal_vmPFC": 0.30,
            "surface_positive_balance": 0.15,
            "anterior_cingulate": 0.10,
            "feature_max": 0.10,
        },
        "critical_features": ["ventral_striatum_nucleus_accumbens", "orbitofrontal_vmPFC"],
        "research_basis": "Consumer-neuroscience papers link ventral striatum/NAcc and vmPFC/OFC to value/desirability/purchase-related signals; TRIBE may not expose subcortical regions, so confidence drops when those are missing.",
    },
    "memory_encoding_potential": {
        "description": "Potential for brand/message encoding and later recall.",
        "engagement_type": "memory / brand recall proxy",
        "weights": {
            "parahippocampal_memory_proxy": 0.30,
            "hippocampus": 0.25,
            "posterior_cingulate_precuneus": 0.20,
            "temporal_semantic_social": 0.15,
            "ventral_visual_object_form": 0.10,
        },
        "critical_features": ["parahippocampal_memory_proxy", "hippocampus", "posterior_cingulate_precuneus"],
        "research_basis": "Memory is a central advertising outcome. This index uses medial-temporal, posterior midline, temporal-semantic and object/form proxies when available.",
    },
    "self_relevance_persuasion": {
        "description": "Potential self-relevance, persuasion, and personal-value processing.",
        "engagement_type": "persuasion / personal relevance proxy",
        "weights": {
            "orbitofrontal_vmPFC": 0.30,
            "posterior_cingulate_precuneus": 0.25,
            "prefrontal_control": 0.20,
            "temporal_semantic_social": 0.15,
            "anterior_cingulate": 0.10,
        },
        "critical_features": ["orbitofrontal_vmPFC", "posterior_cingulate_precuneus"],
        "research_basis": "Neuroforecasting studies often emphasize medial prefrontal/self-related processing for persuasive message response; this index uses vmPFC/OFC and posterior-midline proxies.",
    },
    "social_sharing_potential": {
        "description": "Potential narrative, social cognition, or share-worthiness signal.",
        "engagement_type": "social sharing / narrative resonance proxy",
        "weights": {
            "temporal_semantic_social": 0.35,
            "posterior_cingulate_precuneus": 0.25,
            "orbitofrontal_vmPFC": 0.20,
            "insula_salience": 0.10,
            "prefrontal_control": 0.10,
        },
        "critical_features": ["temporal_semantic_social", "posterior_cingulate_precuneus"],
        "research_basis": "Video-ad liking work highlights social cognition/social-affective response as useful for aggregate liking; this index uses temporal-social and posterior-midline proxies.",
    },
    "affective_salience": {
        "description": "Potential emotional salience or arousal intensity, without naming a discrete emotion.",
        "engagement_type": "emotional salience / arousal proxy",
        "weights": {
            "insula_salience": 0.30,
            "anterior_cingulate": 0.25,
            "amygdala": 0.25,
            "temporal_semantic_social": 0.10,
            "surface_signal_complexity": 0.10,
        },
        "critical_features": ["insula_salience", "anterior_cingulate", "amygdala"],
        "research_basis": "Affective response is a core advertising process. This index estimates salience/arousal-like response rather than claiming happiness/fear/etc.",
    },
    "cognitive_load_informativeness": {
        "description": "How much the ad may drive deliberation, control, reasoning, or utilitarian information processing.",
        "engagement_type": "utilitarian / cognitive processing proxy",
        "weights": {
            "prefrontal_control": 0.35,
            "dorsal_attention_parietal": 0.20,
            "anterior_cingulate": 0.20,
            "temporal_semantic_social": 0.15,
            "feature_std": 0.10,
        },
        "critical_features": ["prefrontal_control", "dorsal_attention_parietal"],
        "research_basis": "Ad research separates information processing/executive function from affective responses; this index represents more analytic or utilitarian engagement.",
    },
    "hedonic_appeal": {
        "description": "Potential fun, pleasantness, aesthetic appeal, or hedonic pull.",
        "engagement_type": "hedonic / liking proxy",
        "weights": {
            "orbitofrontal_vmPFC": 0.30,
            "ventral_striatum_nucleus_accumbens": 0.25,
            "insula_salience": 0.15,
            "visual_cortex": 0.15,
            "ventral_visual_object_form": 0.15,
        },
        "critical_features": ["orbitofrontal_vmPFC", "ventral_striatum_nucleus_accumbens"],
        "research_basis": "Hedonic banner/ad work connects reward, self-relevance and emotion systems to hedonic appeal; this index remains a proxy unless validated against liking data.",
    },
    "novelty_surprise": {
        "description": "Potential novelty, high arousal, or unexpectedness signal.",
        "engagement_type": "novelty / surprise proxy",
        "weights": {
            "surface_high_activation_tail": 0.30,
            "anterior_cingulate": 0.25,
            "dorsal_attention_parietal": 0.15,
            "insula_salience": 0.15,
            "feature_range": 0.15,
        },
        "critical_features": ["surface_high_activation_tail", "anterior_cingulate"],
        "research_basis": "High-response tails, salience, cingulate and attention proxies can indicate stronger response contrast; this does not identify a specific emotion.",
    },
}

def _available_brain_feature_sources(roi_result: Dict[str, Any], feature_dict: Dict[str, float]) -> set:
    available = set((roi_result.get("score_lookup", {}) or {}).keys())
    # Feature statistics are always computed from the brain feature vector.
    available.update([k for k in feature_dict.keys() if k.startswith("feature_") or k.startswith("surface_")])
    return available

def _weighted_proxy_index(feature_dict: Dict[str, float], weights: Dict[str, float], available_sources: set) -> Tuple[float, List[Dict[str, Any]], List[str]]:
    used = []
    missing = []
    total_w = 0.0
    acc = 0.0
    for feature, weight in weights.items():
        if feature not in available_sources:
            missing.append(feature)
            continue
        value = float(feature_dict.get(feature, 0.0))
        if not np.isfinite(value):
            value = 0.0
        value = float(np.clip(value, 0.0, 1.0))
        w = float(weight)
        acc += value * w
        total_w += w
        used.append({"feature": feature, "value": round(value, 4), "weight": round(w, 4)})
    if total_w <= 0:
        return 0.0, used, missing
    return float(np.clip(acc / total_w, 0.0, 1.0)), used, missing

def _proxy_confidence(roi_result: Dict[str, Any], used_components: List[Dict[str, Any]], missing_critical: List[str]) -> str:
    mode = roi_result.get("scoring_mode", "unknown")
    n_used = len(used_components)
    if mode == "unlabeled_surface_profile":
        return "low"
    if n_used >= 3 and not missing_critical:
        return "medium" if mode == "destrieux_surface_atlas" else "high"
    if n_used >= 2:
        return "medium" if len(missing_critical) <= 1 else "low"
    return "low"

def derive_neuromarketing_engagement(roi_result: Dict[str, Any], processed_as: str = "brain_response") -> Dict[str, Any]:
    """Compute transparent, research-informed ad engagement indices from brain-response features.

    These are not an emotion classifier. They answer: what kind of engagement pattern
    the predicted brain activation suggests: attention, memory, valuation/desire, social sharing,
    self-relevance, cognitive/utilitarian processing, hedonic appeal, or novelty/arousal.
    """
    feature_dict = extract_brain_feature_dict(roi_result)
    available_sources = _available_brain_feature_sources(roi_result, feature_dict)
    indices = []
    raw_scores = {}
    for label, spec in NEUROMARKETING_INDEX_SPECS.items():
        score, used, missing = _weighted_proxy_index(feature_dict, spec["weights"], available_sources)
        missing_critical = [x for x in spec.get("critical_features", []) if x in missing]
        raw_scores[label] = score
        indices.append({
            "label": label,
            "score": round(float(score), 4),
            "confidence": _proxy_confidence(roi_result, used, missing_critical),
            "engagement_type": spec.get("engagement_type"),
            "description": spec.get("description"),
            "score_source": "research_informed_brain_feature_formula",
            "score_components": used,
            "missing_features": missing,
            "missing_critical_features": missing_critical,
            "research_basis": spec.get("research_basis"),
            "validation_status": "formula_proxy; validate against real campaign outcomes before using as evidence",
        })

    # Overall index: an interpretable aggregate, not a sales forecast.
    aggregate_weights = {
        "attention_capture": 0.20,
        "desirability_reward": 0.18,
        "memory_encoding_potential": 0.17,
        "self_relevance_persuasion": 0.16,
        "affective_salience": 0.13,
        "social_sharing_potential": 0.10,
        "hedonic_appeal": 0.06,
    }
    overall = 0.0
    denom = 0.0
    for k, w in aggregate_weights.items():
        overall += float(raw_scores.get(k, 0.0)) * float(w)
        denom += float(w)
    overall = float(np.clip(overall / max(denom, 1e-9), 0.0, 1.0))

    ranked = sorted(indices, key=lambda x: float(x["score"]), reverse=True)
    likely_types = []
    for item in ranked:
        score = float(item["score"])
        if score >= 0.66:
            strength = "high"
        elif score >= 0.40:
            strength = "moderate"
        elif score >= 0.20:
            strength = "weak"
        else:
            strength = "very_low"
        likely_types.append({
            "engagement_type": item["engagement_type"],
            "label": item["label"],
            "score": round(score, 4),
            "strength": strength,
            "confidence": item["confidence"],
        })

    # Simple strategic reading from the top pattern.
    top = likely_types[0] if likely_types else None
    if top and top["label"] in {"cognitive_load_informativeness", "attention_capture"} and raw_scores.get("desirability_reward", 0) < 0.35:
        strategic_read = "The ad may be processed more as information/attention than as desire or conversion pull."
    elif top and top["label"] in {"desirability_reward", "hedonic_appeal"}:
        strategic_read = "The ad may create desire/hedonic engagement more than purely informational engagement."
    elif top and top["label"] == "social_sharing_potential":
        strategic_read = "The ad may work through social/narrative resonance and share-worthiness."
    elif top and top["label"] == "memory_encoding_potential":
        strategic_read = "The ad may be relatively stronger for brand/message recall than immediate desire."
    else:
        strategic_read = "The engagement pattern is mixed or low confidence; use A/B tests or labeled outcomes for validation."

    return {
        "available": True,
        "analysis_mode": "computational_neuromarketing_research_informed_indices",
        "evidence_level": "research_informed_formula_proxy_not_validated_kpi",
        "input_used": "TRIBE predicted brain-response ROI/surface features only; no original image emotion classifier is used.",
        "overall_ad_engagement_index": round(overall, 4),
        "neuromarketing_indices": ranked,
        "likely_engagement_types": likely_types[:7],
        "engagement_summary": {
            "top_engagement_type": top,
            "strategic_read": strategic_read,
            "scoring_mode": roi_result.get("scoring_mode", "unknown"),
            "processed_as": processed_as,
        },
        "research_evidence_used": NEUROMARKETING_EVIDENCE,
        "warning": NEUROMARKETING_WARNING_TEXT,
        "limitations": [
            "These are formula-based proxy indices, not a guaranteed campaign KPI predictor.",
            "TRIBE output is predicted neural activity, not observed human fMRI from your audience.",
            "Subcortical reward/affect regions may be absent from TRIBE cortical output; related indices become lower-confidence proxies.",
            "For claims about sales, recall, liking, shares, or purchase intent, validate against real campaign outcomes or human studies.",
        ],
    }

def derive_computational_neuromarketing(roi_result: Dict[str, Any], processed_as: str = "brain_response") -> Dict[str, Any]:
    """Return research-informed neuromarketing proxy indices only."""
    return derive_neuromarketing_engagement(roi_result, processed_as=processed_as)

print("Brain feature extraction and neuromarketing index functions ready.")


Brain feature extraction and neuromarketing index functions ready.


## 8. Visualization and analysis payload

In [12]:
import matplotlib.pyplot as plt

def generate_brain_visualization(roi_result: Dict[str, Any], output_path: Path, title: str) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    region_scores = [
        x for x in roi_result.get("region_scores", [])
        if x.get("available") and x.get("activation") is not None
    ]
    region_scores = sorted(region_scores, key=lambda x: float(x.get("activation", 0)), reverse=True)

    if region_scores:
        labels = [x["region"] for x in region_scores]
        vals = [float(x["activation"]) for x in region_scores]
        fig = plt.figure(figsize=(12, max(5, 0.42 * len(labels) + 2)))
        ax = fig.add_subplot(111)
        y = np.arange(len(labels))
        ax.barh(y, vals)
        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.invert_yaxis()
        ax.set_xlim(0, 1)
        ax.set_xlabel("Normalized predicted activation proxy")
        ax.set_title(title)
        ax.grid(axis="x", alpha=0.25)
        fig.tight_layout()
        fig.savefig(output_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        free_runtime_memory()
        return output_path

    stat = roi_result.get("stat_map")
    stat = np.asarray(stat if stat is not None else [], dtype=np.float32).reshape(-1)
    fig = plt.figure(figsize=(12, 4))
    ax = fig.add_subplot(111)
    if stat.size:
        sample = stat
        if stat.size > 1000:
            idx = np.linspace(0, stat.size - 1, 1000).astype(int)
            sample = stat[idx]
        ax.plot(sample)
    ax.set_title(title + " — fallback cortical vertex trace")
    ax.set_xlabel("Sampled cortical vertex index")
    ax.set_ylabel("Predicted activity")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    free_runtime_memory()
    return output_path

def generate_neuromarketing_visualization(indices: List[Dict[str, Any]], output_path: Path, title: str) -> Optional[Path]:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    scores = [x for x in indices if x.get("score") is not None]
    if not scores:
        return None
    scores = sorted(scores, key=lambda x: float(x.get("score", 0)), reverse=True)
    labels = [x.get("label", "index") for x in scores]
    vals = [float(x.get("score", 0.0)) for x in scores]
    fig = plt.figure(figsize=(12, max(5, 0.45 * len(labels) + 2)))
    ax = fig.add_subplot(111)
    y = np.arange(len(labels))
    ax.barh(y, vals)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.set_xlabel("Research-informed engagement proxy score")
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    free_runtime_memory()
    return output_path


def analyze_uploaded_ad_sync(ad_file_path: Path, campaign_name: Optional[str], normalize_scores: bool, run_dir: Path) -> Dict[str, Any]:
    """
    Clean pipeline:
    1. Validate the uploaded ad image.
    2. Convert it to a tiny silent video for TRIBE.
    3. Run TRIBE and extract predicted brain-response ROI/surface features.
    4. Compute research-informed neuromarketing engagement indices.

    This function does not infer emotions, train models, load decoders, or create mapping JSON files.
    """
    image_info = validate_image_file(ad_file_path)

    inference_result = run_tribe_inference_for_upload(ad_file_path, run_dir)
    raw_output = inference_result["raw_output"]
    processed_as = inference_result["processed_as"]
    temporary_video_path = inference_result.get("temporary_video_path")

    try:
        roi_result = compute_roi_scores(raw_output, normalize_scores=normalize_scores)
    finally:
        try:
            del raw_output
            inference_result["raw_output"] = None
        except Exception:
            pass
        free_runtime_memory()

    brain_features = extract_brain_feature_dict(roi_result)
    brain_feature_summary = summarize_brain_features(brain_features)
    neuromarketing = derive_computational_neuromarketing(roi_result, processed_as=processed_as)

    result_id = uuid.uuid4().hex[:12]
    brain_vis_path = RESULTS_DIR / f"brain_activation_{result_id}.png"
    generate_brain_visualization(
        roi_result,
        brain_vis_path,
        title=f"TRIBE v2 predicted brain-response activation: {campaign_name or 'advertisement'}",
    )

    neuromarketing_vis_path = RESULTS_DIR / f"neuromarketing_engagement_{result_id}.png"
    neuromarketing_chart = generate_neuromarketing_visualization(
        neuromarketing.get("neuromarketing_indices", []),
        neuromarketing_vis_path,
        title=f"Computational neuromarketing engagement indices: {campaign_name or 'advertisement'}",
    )

    # Do not keep large stat maps inside response-side objects.
    roi_result["stat_map"] = None
    free_runtime_memory()

    base = current_base_url().rstrip("/") if "current_base_url" in globals() else "http://localhost:8000"

    payload = {
        "status": "success",
        "campaign_name": campaign_name or "untitled_campaign",
        "input_type": "image",
        "processed_as": processed_as,
        "analysis_mode": neuromarketing.get("analysis_mode", "computational_neuromarketing_research_informed_indices"),
        "model": "TRIBE v2",
        "model_status": {
            "loaded": MODEL_STATUS.get("loaded"),
            "source": MODEL_STATUS.get("source"),
            "device": MODEL_STATUS.get("device"),
            "checkpoint": MODEL_STATUS.get("checkpoint"),
            "error": MODEL_STATUS.get("error"),
        },
        "prediction_shape": inference_result.get("prediction_shape"),
        "events_count": inference_result.get("events_count"),
        "warning": neuromarketing.get("warning", BRAIN_FEATURE_WARNING_TEXT),
        "required_wording": SYSTEM_LABEL,
        "image_info": image_info,
        "scoring_mode": roi_result.get("scoring_mode"),
        "roi_metadata": roi_result.get("metadata", {}),
        "brain_features": brain_features,
        "brain_feature_summary": brain_feature_summary,
        "brain_region_scores": [
            {
                "region": item["region"],
                "activation": item.get("activation"),
                "confidence": item.get("confidence", "low"),
                "interpretation": item.get("interpretation", ""),
                "available": item.get("available", False),
            }
            for item in roi_result.get("region_scores", [])
        ],
        "neuromarketing_analysis": {
            "analysis_mode": neuromarketing.get("analysis_mode"),
            "evidence_level": neuromarketing.get("evidence_level"),
            "overall_ad_engagement_index": neuromarketing.get("overall_ad_engagement_index"),
            "likely_engagement_types": neuromarketing.get("likely_engagement_types", []),
            "engagement_summary": neuromarketing.get("engagement_summary", {}),
            "warning": neuromarketing.get("warning"),
        },
        "neuromarketing_indices": neuromarketing.get("neuromarketing_indices", []),
        "neuromarketing_evidence_used": neuromarketing.get("research_evidence_used", []),
        "brain_activation_image_url": f"{base}/results/{brain_vis_path.name}",
        "neuromarketing_chart_url": f"{base}/results/{neuromarketing_vis_path.name}" if neuromarketing_chart else None,
        "limitations": neuromarketing.get("limitations", []) + [
            "The uploaded ad is only used as a stimulus for TRIBE; no separate image emotion classifier is used.",
            "TRIBE v2 predictions usually live on a cortical surface; subcortical areas are unavailable unless explicitly returned.",
            "When possible, cortical vertex outputs are converted to broad ROI proxies with Nilearn's Destrieux surface atlas; otherwise an unlabeled low-confidence surface-profile fallback is used.",
            "Static images are downscaled, converted to a one-second silent video, and passed to TRIBE through video_path.",
        ],
        "_internal_cleanup_paths": {
            "temporary_video_path": str(temporary_video_path) if temporary_video_path else None,
            "run_dir": str(run_dir),
        },
        "cleanup": {
            "uploaded_image_deleted": False,
            "temporary_video_deleted": False,
            "intermediate_files_deleted": False,
            "retained_files": [str(x) for x in [brain_vis_path, neuromarketing_chart] if x is not None],
        },
        "memory_after_analysis": memory_snapshot(),
    }

    try:
        del roi_result, inference_result, neuromarketing, brain_features, brain_feature_summary
    except Exception:
        pass
    free_runtime_memory()
    return payload

print("Clean neuromarketing analysis function ready.")


Clean neuromarketing analysis function ready.


## 9. FastAPI backend

In [13]:
import json
import uuid
from pathlib import Path
from typing import Optional
from threading import Lock

from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles


app = FastAPI(
    title="Advertisement Brain-Response Neuromarketing API using TRIBE v2",
    version="7.0-clean-neuromarketing-inference",
    description="Clean inference API: TRIBE-predicted brain-response features plus neuromarketing engagement indices. No training, decoder, PNG-only fallback, or static research-mapping file.",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

app.mount("/results", StaticFiles(directory=str(RESULTS_DIR)), name="results")
app.state.public_url = None
INFERENCE_LOCK = Lock()

def current_base_url() -> str:
    return app.state.public_url or "http://localhost:8000"

@app.get("/")
def root():
    return {
        "status": "running",
        "model": "TRIBE v2",
        "device": MODEL_STATUS.get("device", DEVICE),
        "model_loaded": bool(MODEL_STATUS.get("loaded")),
        "model_source": MODEL_STATUS.get("source"),
        "model_error": MODEL_STATUS.get("error"),
        "public_url": app.state.public_url,
        "warning": NEUROMARKETING_WARNING_TEXT,
        "required_wording": SYSTEM_LABEL,
        "endpoints": {
            "analyze_ad_neuromarketing": "/analyze-ad",
            "health": "/health",
            "memory": "/memory",
            "routes": "/routes",
            "docs": "/docs",
            "openapi": "/openapi.json",
            "results": "/results/{filename}",
        },
        "usage": "POST /analyze-ad with an ad image. The response contains brain features, neuromarketing indices, and charts only.",
        "disabled_features": [
            "emotion detection",
            "CSV training",
            "decoder/joblib loading",
            "static research-mapping JSON",
            "PNG-only brain-image heuristic endpoint",
        ],
        "oom_safe_limits": {
            "MAX_UPLOAD_MB": MAX_UPLOAD_MB,
            "MAX_IMAGE_PIXELS": MAX_IMAGE_PIXELS,
            "MAX_IMAGE_SIDE": MAX_IMAGE_SIDE,
            "VIDEO_DURATION_SECONDS": VIDEO_DURATION_SECONDS,
            "VIDEO_FPS": VIDEO_FPS,
        },
        "scoring": {
            "uses_destrieux_atlas_when_possible": True,
            "uses_original_image_for_emotion_classifier": False,
            "static_research_mapping_used": False,
            "engagement_index_labels": list(NEUROMARKETING_INDEX_SPECS.keys()),
        },
    }

@app.get("/health")
def health():
    return {
        "ok": True,
        "status": "running",
        "model_loaded": bool(MODEL_STATUS.get("loaded")),
        "model_error": MODEL_STATUS.get("error"),
        "device": MODEL_STATUS.get("device", DEVICE),
        "analysis_mode": "clean_neuromarketing_inference",
        "memory": memory_snapshot(),
    }

@app.get("/memory")
def memory():
    free_runtime_memory()
    return memory_snapshot()

@app.get("/routes")
def routes():
    return {
        "routes": sorted([
            {
                "path": route.path,
                "name": route.name,
                "methods": sorted(list(route.methods or [])),
            }
            for route in app.routes
        ], key=lambda item: item["path"]),
        "has_analyze_ad": any(route.path == "/analyze-ad" for route in app.routes),
    }

async def save_upload_to_disk(upload: UploadFile, dst: Path, max_bytes: int = MAX_UPLOAD_BYTES) -> Path:
    dst.parent.mkdir(parents=True, exist_ok=True)
    total = 0
    with open(dst, "wb") as f:
        while True:
            chunk = await upload.read(1024 * 1024)
            if not chunk:
                break
            total += len(chunk)
            if total > max_bytes:
                raise ValueError(f"Uploaded file is too large. Limit is {MAX_UPLOAD_MB} MB.")
            f.write(chunk)
    if not dst.exists() or dst.stat().st_size == 0:
        raise ValueError("Uploaded file was empty.")
    return dst

@app.post("/analyze-ad")
async def analyze_ad(
    file: UploadFile = File(...),
    campaign_name: Optional[str] = Form(None),
    normalize_scores: bool = Form(True),
    cleanup_all: bool = Form(True),
):
    run_id = uuid.uuid4().hex
    run_dir = TEMP_DIR / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    suffix = Path(file.filename or "ad.png").suffix.lower() or ".png"
    uploaded_path = run_dir / f"{safe_stem(file.filename or 'ad')}{suffix}"
    temporary_video_path = None
    payload = None
    status_code = 200

    try:
        if MODEL is None or not MODEL_STATUS.get("loaded"):
            raise RuntimeError(f"TRIBE v2 model is not loaded: {MODEL_STATUS.get('error')}")

        await save_upload_to_disk(file, uploaded_path)

        # Prevent overlapping GPU/CPU inference in a single Kaggle runtime.
        with INFERENCE_LOCK:
            free_runtime_memory()
            payload = analyze_uploaded_ad_sync(uploaded_path, campaign_name, normalize_scores, run_dir)
            free_runtime_memory()

        internal = payload.pop("_internal_cleanup_paths", {})
        if internal.get("temporary_video_path"):
            temporary_video_path = Path(internal["temporary_video_path"])
    except ValueError as exc:
        status_code = 400
        payload = {
            "status": "error",
            "error_type": "invalid_image",
            "message": str(exc),
            "warning": NEUROMARKETING_WARNING_TEXT,
            "limits": {
                "MAX_UPLOAD_MB": MAX_UPLOAD_MB,
                "MAX_IMAGE_PIXELS": MAX_IMAGE_PIXELS,
                "MAX_IMAGE_SIDE": MAX_IMAGE_SIDE,
            },
        }
    except RuntimeError as exc:
        message = str(exc)
        status_code = 503 if "not loaded" in message.lower() or "unavailable" in message.lower() else 500
        payload = {
            "status": "error",
            "error_type": "model_unavailable_or_inference_failed",
            "message": message,
            "model_status": {
                "loaded": MODEL_STATUS.get("loaded"),
                "source": MODEL_STATUS.get("source"),
                "device": MODEL_STATUS.get("device"),
                "checkpoint": MODEL_STATUS.get("checkpoint"),
                "error": MODEL_STATUS.get("error"),
            },
            "memory": memory_snapshot(),
            "warning": NEUROMARKETING_WARNING_TEXT,
        }
    except Exception as exc:
        status_code = 500
        payload = {
            "status": "error",
            "error_type": "unexpected_error",
            "message": repr(exc),
            "memory": memory_snapshot(),
            "warning": NEUROMARKETING_WARNING_TEXT,
        }
    finally:
        uploaded_deleted = delete_path(uploaded_path)
        video_deleted = delete_path(temporary_video_path)
        run_dir_deleted = delete_path(run_dir) if cleanup_all else False
        free_runtime_memory()

        if isinstance(payload, dict):
            payload["cleanup"] = {
                "uploaded_image_deleted": bool(uploaded_deleted or not uploaded_path.exists()),
                "temporary_video_deleted": bool(video_deleted or temporary_video_path is None or not Path(temporary_video_path).exists()),
                "intermediate_files_deleted": bool(run_dir_deleted or not run_dir.exists()),
                "cleanup_all_requested": bool(cleanup_all),
                "retained_files": payload.get("cleanup", {}).get("retained_files", []),
            }

    return JSONResponse(status_code=status_code, content=payload)

registered_paths = sorted({route.path for route in app.routes})
assert "/analyze-ad" in registered_paths, "FastAPI route /analyze-ad was not registered."

print("Clean neuromarketing FastAPI app created.")
print("Registered routes:", registered_paths)
print(json.dumps(root(), indent=2))


Clean neuromarketing FastAPI app created.
Registered routes: ['/', '/analyze-ad', '/docs', '/docs/oauth2-redirect', '/health', '/memory', '/openapi.json', '/redoc', '/results', '/routes']
{
  "status": "running",
  "model": "TRIBE v2",
  "device": "cuda",
  "model_loaded": true,
  "model_source": "tribev2.TribeModel",
  "model_error": null,
  "public_url": null,
  "warning": "Neuromarketing indices are research-informed proxy formulas over predicted brain-response features. They are not a validated emotion reader, not observed fMRI, and not a guaranteed KPI forecast. For production claims, validate these proxy indices against real campaign outcomes, A/B tests, or human studies.",
  "required_wording": "Predicted neural-response proxies derived from TRIBE v2 outputs.",
  "endpoints": {
    "analyze_ad_neuromarketing": "/analyze-ad",
    "health": "/health",
    "memory": "/memory",
    "routes": "/routes",
    "docs": "/docs",
    "openapi": "/openapi.json",
    "results": "/results/{fi

## 10. Start server and ngrok tunnel

Requires a Kaggle secret/environment variable named `NGROK_AUTH_TOKEN` or `NGROK_AUTHTOKEN`.

This fixed starter verifies that `/analyze-ad` is registered before reusing an existing localhost server. If an old server is still occupying port `8000`, it automatically starts the corrected app on the next available port and points ngrok to that port.


In [14]:
import os
import time
import socket
import threading
import requests
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

PUBLIC_API_URL = None
API_PORT = int(os.environ.get("API_PORT", "8000"))
LOCAL_API_URL = f"http://localhost:{API_PORT}"
SERVER_THREAD = None

def tcp_port_is_open(port: int, host: str = "127.0.0.1", timeout: float = 0.5) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0

def server_has_analyze_route(base_url: str) -> bool:
    try:
        response = requests.get(f"{base_url}/openapi.json", timeout=2)
        if response.status_code != 200:
            return False
        paths = response.json().get("paths", {})
        return "/analyze-ad" in paths and "post" in paths.get("/analyze-ad", {})
    except Exception:
        return False

def wait_for_analyze_route(base_url: str, seconds: int = 60) -> None:
    last_error = None
    for _ in range(seconds):
        try:
            if server_has_analyze_route(base_url):
                return
            last_error = "server responded, but /analyze-ad was not in OpenAPI paths yet"
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(1)
    raise RuntimeError(f"FastAPI server did not expose POST /analyze-ad at {base_url}. Last check: {last_error}")

def choose_api_port(preferred_port: int = 8000, max_tries: int = 20):
    """Return (port, can_reuse_existing_server)."""
    for port in range(preferred_port, preferred_port + max_tries):
        base_url = f"http://localhost:{port}"

        # Reuse only if the running server is already the corrected app.
        if server_has_analyze_route(base_url):
            return port, True

        # If some old server or another process is occupying the port, skip it.
        if tcp_port_is_open(port):
            print(f"Port {port} is occupied, but it does not expose POST /analyze-ad. Trying next port...")
            continue

        return port, False

    raise RuntimeError(f"No usable port found in range {preferred_port}-{preferred_port + max_tries - 1}.")

def start_uvicorn_background():
    global SERVER_THREAD, API_PORT, LOCAL_API_URL

    API_PORT, reuse_existing = choose_api_port(API_PORT)
    LOCAL_API_URL = f"http://localhost:{API_PORT}"

    if reuse_existing:
        print(f"A corrected FastAPI server is already responding at {LOCAL_API_URL}.")
        return SERVER_THREAD

    if SERVER_THREAD is not None and SERVER_THREAD.is_alive():
        print("A server thread exists, but the route check failed. Starting a fresh server on a different port.")

    def run():
        uvicorn.run(app, host="0.0.0.0", port=API_PORT, log_level="info")

    SERVER_THREAD = threading.Thread(target=run, daemon=True)
    SERVER_THREAD.start()

    wait_for_analyze_route(LOCAL_API_URL, seconds=60)
    print(f"Local FastAPI server is responding with POST /analyze-ad at {LOCAL_API_URL}.")
    return SERVER_THREAD

def start_ngrok_tunnel():
    global PUBLIC_API_URL

    token, token_source = get_secret_or_env(["NGROK_AUTH_TOKEN", "NGROK_AUTHTOKEN"])
    if not token:
        raise RuntimeError("Missing ngrok auth token. Add Kaggle Secret NGROK_AUTH_TOKEN or environment variable NGROK_AUTH_TOKEN.")

    ngrok.set_auth_token(token)
    print("Loaded ngrok token from", token_source)

    try:
        ngrok.kill()
    except Exception:
        pass

    tunnel = ngrok.connect(API_PORT, "http")
    PUBLIC_API_URL = tunnel.public_url
    app.state.public_url = PUBLIC_API_URL
    return PUBLIC_API_URL

start_uvicorn_background()
PUBLIC_API_URL = start_ngrok_tunnel()

print("=" * 60)
print("FastAPI Server Running")
print(f"Local URL:  {LOCAL_API_URL}")
print(f"Public URL: {PUBLIC_API_URL}")
print("=" * 60)
print("Health check:")
print(f"curl {PUBLIC_API_URL}/health")
print("\nRoute check:")
print(f"curl {PUBLIC_API_URL}/routes")
print("\nOpenAPI route check:")
print(f"curl {PUBLIC_API_URL}/openapi.json")
print("\nAnalyze ad example:")
print(f"""curl -X POST "{PUBLIC_API_URL}/analyze-ad" \\
  -F "file=@/path/to/ad.jpg" \\
  -F "campaign_name=World Cup Campaign" \\
  -F "normalize_scores=true" \\
  -F "cleanup_all=true"
""")


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:39760 - "GET /openapi.json HTTP/1.1" 200 OK
Local FastAPI server is responding with POST /analyze-ad at http://localhost:8000.
Loaded ngrok token from kaggle_secret:NGROK_AUTH_TOKEN                                              
FastAPI Server Running
Local URL:  http://localhost:8000
Public URL: https://muck-suction-matchbook.ngrok-free.dev
Health check:
curl https://muck-suction-matchbook.ngrok-free.dev/health

Route check:
curl https://muck-suction-matchbook.ngrok-free.dev/routes

OpenAPI route check:
curl https://muck-suction-matchbook.ngrok-free.dev/openapi.json

Analyze ad example:
curl -X POST "https://muck-suction-matchbook.ngrok-free.dev/analyze-ad" \
  -F "file=@/path/to/ad.jpg" \
  -F "campaign_name=World Cup Campaign" \
  -F "normalize_scores=true" \
  -F "cleanup_all=true"



Extract audio from video events:   0%|          | 0/1 [00:00<?, ?it/s]

MoviePy - Writing audio in /kaggle/working/ad_api_temp/c6bdd3b0c7ca48789bd7f2bdf2cbd336/processing/10245_1522c137_silent.wav



Extract audio from video events: 100%|██████████| 1/1 [00:00<00:00, 14.87it/s]
/usr/local/lib/python3.12/dist-packages/neuralset/events/transforms/audio.py:56: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat([events, pd.DataFrame(events_to_add)], ignore_index=True)
/usr/local/lib/python3.12/dist-packages/neuralset/events/utils.py:134: UserWarning: The events dataframe contains an `Index` column. This is dangerous, please add drop=True in calls to df.reset_index(). Dropping it automatically.
  warnings.warn(msg)


MoviePy - Done.


... (8 outputs left out)

Loading weights:   0%|          | 0/843 [00:00<?, ?it/s]

video_preprocessor_config.json: 0.00B [00:00, ?B/s]

2026-06-09 12:58:36 - DEBUG - neuralset.extractors.video:277 - Loaded Video (duration 1.0s at 1.0fps, shape (512, 512)):
/kaggle/working/ad_api_temp/c6bdd3b0c7ca48789bd7f2bdf2cbd336/processing/10245_1522c137_silent.mp4
Encoding video: 100%|██████████| 2/2 [00:29<00:00, 14.98s/it]
[12:59:06 INFO] Preparing extractor: subject_id
2026-06-09 12:59:06 - WARNING - neuralset.extractors.base:824 - LabelEncoder has only found one label: {'default'}. This was probably not intended.
[12:59:07 INFO] Building dataloader for split all
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
100%|██████████| 

INFO:     103.91.52.5:0 - "POST /analyze-ad HTTP/1.1" 200 OK


## Example requests

Get server status:

```bash
curl "${PUBLIC_API_URL}/"
```

```bash
curl "${PUBLIC_API_URL}/health"
```

Analyze an ad image:

```bash
curl -X POST "${PUBLIC_API_URL}/analyze-ad" \
  -F "file=@/path/to/ad.png" \
  -F "campaign_name=World Cup Campaign" \
  -F "normalize_scores=true" \
  -F "cleanup_all=true"
```

`campaign_name` is optional. It is only used as metadata and chart title text.

Expected output shape:

```json
{
  "status": "success",
  "analysis_mode": "computational_neuromarketing_research_informed_indices",
  "brain_features": {},
  "brain_feature_summary": {},
  "brain_region_scores": [],
  "neuromarketing_analysis": {
    "overall_ad_engagement_index": 0.53,
    "likely_engagement_types": []
  },
  "neuromarketing_indices": [],
  "brain_activation_image_url": "https://.../results/brain_activation_xxx.png",
  "neuromarketing_chart_url": "https://.../results/neuromarketing_engagement_xxx.png"
}
```

## Removed from this clean notebook

- `RESEARCH_MAPPING` file generation
- brain-region-to-emotion JSON mapping
- emotion score output
- `/detect-emotion-from-brain-image` endpoint
- PNG-only color/brightness heuristic
- CSV training helpers
- trained decoder loading
- duplicate fallback proxy score output
- compressed intermediate stat-map saving

## OOM troubleshooting

After an OOM restart, rerun the notebook from the first cell. Do not only rerun the API cell.

For lower memory usage on Kaggle, set this before running the notebook:

```python
import os
os.environ["MAX_IMAGE_SIDE"] = "384"
```

## What this notebook can and cannot claim

It can say: "TRIBE predicted this brain-response feature profile for the stimulus."

It can say: "This profile has higher attention/memory/desire/social-sharing proxy indices," with clear proxy wording.

It should not say: "this ad made the viewer feel happy/sad/angry."

It should not present the proxy indices as guaranteed sales, CTR, recall, liking, or purchase-intent predictions.
